# Private RLHF: Qualitative Results

This notebook generates the qualitative comparisons and Best-of-`N` outputs used in the paper. It reads `master_results.csv` and the saved artifacts created by the training notebook.

The first workflow compares the proposed method with the policy baselines. The second workflow generates the mixed-temperature candidate pools used for the Best-of-`N` illustration. Each workflow writes its outputs to a separate CSV and candidate directory under `ROOT`.

## Colab setup

Mount Drive and point `ROOT` to the folder created by the training notebook.

In [ ]:
!pip -q install "transformers>=4.40.0" datasets peft accelerate safetensors

import os
from google.colab import drive

drive.mount("/content/drive")

# Change this line only if the training artifacts were saved elsewhere.
ROOT = "/content/drive/MyDrive/Private_Finetuning"
MASTER_CSV = os.path.join(ROOT, "master_results.csv")
MODEL_ID = "google/gemma-2b-it"

print("ROOT       =", ROOT)
print("MASTER_CSV =", MASTER_CSV)

## Across-method qualitative comparison

This workflow generates outputs from the proposed method, DP-DPO, and the split DP policy-optimization baseline for the fixed prompt set.

In [ ]:
import os, re, json, time, csv, hashlib, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

try:
    from peft import PeftModel
except Exception as e:
    raise RuntimeError("peft is required. Install/ensure it exists (pip install peft).") from e

## Configuration

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "GPU is required."

# Use your existing globals if present
ROOT = globals().get("ROOT", "/content/drive/MyDrive/Private_Finetuning")
MASTER_CSV = globals().get("MASTER_CSV", os.path.join(ROOT, "master_results.csv"))
MODEL_ID = globals().get("MODEL_ID", "google/gemma-2b-it")

# Training seed used to select saved artifacts
ARTIFACT_SEED = 11

# Privacy and candidate-set grids
EPS_LIST = [0.5, 1.0, 2.0]
N_GRID  = [2, 4, 8, 16, 32]

# Fixed prompts identified by hash
# Prompt hashes used in the qualitative analysis
TARGET_PROMPT_HASHES = [
    "6af5546e8a73",  # Candidate 17
    "e0966f2cec3d",  # Candidate 20
    "4461aba094ae",  # Candidate 7
    "bc1d9cb75240",  # Candidate 18
    "89a6a5b5c6a2",  # Candidate 16
]

# decoding (fixed across all methods)
MAX_NEW_TOKENS = 160
TEMPERATURE = 0.8
TOP_P = 0.9

# Dataset slice used in the training pipeline
N_TOTAL = 40000
TEST_FRAC = 0.2

# generation seed base (deterministic reproducibility)
GEN_SEED_BASE = 2026

# output locations
QUAL_CSV = os.path.join(ROOT, "qualitative_generations.csv")
CAND_DIR = os.path.join(ROOT, "qual_candidates")  # candidate dumps for ours
os.makedirs(ROOT, exist_ok=True)
os.makedirs(CAND_DIR, exist_ok=True)

# Optional: if True, prepend a safety instruction to prompts (kept OFF by default)
SAFETY_GUARD = False
SAFETY_PREFIX = (
    "Human: For safety and privacy, do NOT provide personal addresses, phone numbers, passwords, "
    "or any instructions for wrongdoing. If asked, refuse and offer safe alternatives.\n\nAssistant:\n"
)

print("DEVICE       =", DEVICE)
print("ROOT         =", ROOT)
print("MASTER_CSV   =", MASTER_CSV)
print("MODEL_ID     =", MODEL_ID)
print("QUAL_CSV     =", QUAL_CSV)
print("CAND_DIR     =", CAND_DIR)

## Utility functions

In [ ]:
PROMPT_PAT = re.compile(r"\n\nAssistant:\s*")

def extract_prompt_prefix(text: str) -> str:
    ms = list(PROMPT_PAT.finditer(text))
    if not ms:
        return ""
    return text[: ms[-1].end()]

def sha1_12(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:12]

def now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def cleanup(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    torch.cuda.empty_cache()

def load_master_df(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"MASTER_CSV not found: {path}")
    return pd.read_csv(path)

def latest_ok_row(df: pd.DataFrame, *, method: str, eps: float, seed: int, split_role: str = None):
    sub = df[
        (df["method"].astype(str) == str(method)) &
        (pd.to_numeric(df["epsilon_target"], errors="coerce") == float(eps)) &
        (pd.to_numeric(df["seed"], errors="coerce") == int(seed)) &
        (df["train_status"].astype(str) == "ok") &
        (df["eval_status"].astype(str) == "ok")
    ].copy()
    if split_role is not None:
        sub = sub[sub["split_role"].astype(str) == str(split_role)]
    if sub.empty:
        return None
    sub["timestamp_dt"] = pd.to_datetime(sub["timestamp"], errors="coerce")
    sub = sub.sort_values(["timestamp_dt", "run_id"], ascending=True)
    return sub.iloc[-1]

def ensure_csv_schema(path: str, cols: list):
    if os.path.exists(path):
        df = pd.read_csv(path)
        for c in cols:
            if c not in df.columns:
                df[c] = np.nan
        df.to_csv(path, index=False, quoting=csv.QUOTE_ALL)
    else:
        pd.DataFrame(columns=cols).to_csv(path, index=False, quoting=csv.QUOTE_ALL)

def load_existing_keys(path: str, key_cols: list):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    return set(tuple(str(r.get(c, "")) for c in key_cols) for _, r in df.iterrows())

def append_rows_csv(path: str, rows: list, cols: list):
    if not rows:
        return
    df_new = pd.DataFrame(rows, columns=cols)
    if os.path.exists(path):
        df_old = pd.read_csv(path)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(path, index=False, quoting=csv.QUOTE_ALL)

@torch.no_grad()
def generate_completion(model, tokenizer, prompt: str, seed: int) -> str:
    torch.manual_seed(seed)
    np.random.seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_new_tokens=MAX_NEW_TOKENS,
        pad_token_id=tokenizer.eos_token_id,
    )
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full[len(prompt):].lstrip()

def load_rm_head_only(artifact_dir: str, tokenizer, dtype=torch.float16):
    meta_path = os.path.join(artifact_dir, "rm_head_meta.json")
    head_path = os.path.join(artifact_dir, "rm_head.pt")
    if not (os.path.exists(meta_path) and os.path.exists(head_path)):
        raise FileNotFoundError(f"RM files not found under: {artifact_dir}")

    with open(meta_path, "r") as f:
        meta = json.load(f)
    head_attr = meta["head_attr"]
    model_id = meta["model_id"]

    rm = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=1, torch_dtype=dtype
    ).to(DEVICE)
    rm.config.pad_token_id = tokenizer.pad_token_id

    for p in rm.parameters():
        p.requires_grad_(False)

    head = getattr(rm, head_attr)
    sd = torch.load(head_path, map_location="cpu")
    head.load_state_dict(sd)

    rm.eval()
    return rm

@torch.no_grad()
def rm_score_texts(rm, tokenizer, prompt: str, completions: list, max_len: int = 256) -> np.ndarray:
    texts = [prompt + c for c in completions]
    enc = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=max_len
    ).to(DEVICE)
    s = rm(**enc).logits.squeeze(-1).detach().float().cpu().numpy()
    return s

@torch.no_grad()
def ours_best_of_n(pi0, rm, tokenizer, prompt: str, N: int, seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outs = pi0.generate(
        **inputs,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_new_tokens=MAX_NEW_TOKENS,
        num_return_sequences=N,
        pad_token_id=tokenizer.eos_token_id,
    )
    full_texts = tokenizer.batch_decode(outs, skip_special_tokens=True)
    comps = [t[len(prompt):].lstrip() for t in full_texts]
    scores = rm_score_texts(rm, tokenizer, prompt, comps, max_len=256)
    best_idx = int(np.argmax(scores))
    return comps, scores, best_idx

def save_candidates_json(path: str, payload: dict):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)

## Load the tokenizer and base policy

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

pi0 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
pi0.eval()

## Recover fixed evaluation prompts

In [ ]:
target_set = set(TARGET_PROMPT_HASHES)
found = {}  # hash -> prompt_text

raw = load_dataset("Anthropic/hh-rlhf", split=f"train[:{N_TOTAL}]")
split = raw.train_test_split(test_size=float(TEST_FRAC), shuffle=True, seed=int(ARTIFACT_SEED))
test_raw = split["test"]

for i in range(len(test_raw)):
    p = extract_prompt_prefix(test_raw[i]["chosen"])
    if not p:
        continue
    h = sha1_12(p)
    if h in target_set and h not in found:
        found[h] = p
    if len(found) == len(target_set):
        break

missing = sorted(list(target_set - set(found.keys())))
if missing:
    # If not found in test split, try scanning train split too (still within the same 40k slice)
    for i in range(len(split["train"])):
        p = extract_prompt_prefix(split["train"][i]["chosen"])
        if not p:
            continue
        h = sha1_12(p)
        if h in target_set and h not in found:
            found[h] = p
        if len(found) == len(target_set):
            break

missing = sorted(list(target_set - set(found.keys())))
if missing:
    raise RuntimeError(
        "Could not recover all target prompts by hash. Missing hashes: "
        + ", ".join(missing)
        + "\n(If this happens, set N_TOTAL/SEED/TEST_FRAC to match your earlier sampling exactly, "
          "or paste the 5 prompt strings manually.)"
    )

# Fix prompt order to the hash list order
prompts = [(h, found[h]) for h in TARGET_PROMPT_HASHES]

print("\n[OK] Loaded 5 fixed prompts by hash (showing only short previews):")
for j, (h, p) in enumerate(prompts):
    preview = p.replace("\n", " ")[:180]
    print(f"  prompt_id={j} hash={h} preview='{preview}...'")

# optional safety prefix
def maybe_guard_prompt(p: str) -> str:
    if not SAFETY_GUARD:
        return p
    # Keep HH-style: prefix instruction as a new "Human:" block before original prompt
    return SAFETY_PREFIX + p

## Locate trained artifacts

In [ ]:
m = load_master_df(MASTER_CSV)

artifact = {"rm": {}, "dpo": {}, "ppo": {}}
for eps in EPS_LIST:
    r = latest_ok_row(m, method="dp_rm_postproc", eps=eps, seed=ARTIFACT_SEED, split_role="full")
    if r is None:
        raise RuntimeError(f"Missing dp_rm_postproc artifact for eps={eps}, seed={ARTIFACT_SEED}")
    artifact["rm"][eps] = {"run_id": str(r["run_id"]), "dir": str(r["artifact_dir"])}

    r = latest_ok_row(m, method="dp_dpo", eps=eps, seed=ARTIFACT_SEED, split_role="full")
    if r is None:
        raise RuntimeError(f"Missing dp_dpo artifact for eps={eps}, seed={ARTIFACT_SEED}")
    artifact["dpo"][eps] = {"run_id": str(r["run_id"]), "dir": str(r["artifact_dir"])}

    r = latest_ok_row(m, method="dp_ppo_like_split", eps=eps, seed=ARTIFACT_SEED, split_role="po_half")
    if r is None:
        raise RuntimeError(f"Missing dp_ppo_like_split artifact for eps={eps}, seed={ARTIFACT_SEED}")
    artifact["ppo"][eps] = {"run_id": str(r["run_id"]), "dir": str(r["artifact_dir"])}

print("\n[OK] Artifacts located:")
for eps in EPS_LIST:
    print(f"  eps={eps} | RM={artifact['rm'][eps]['run_id']} | DPO={artifact['dpo'][eps]['run_id']} | PPO={artifact['ppo'][eps]['run_id']}")

## Load policy adapters and reward models

In [ ]:
COLS = [
    "timestamp",
    "prompt_id", "prompt_hash",
    "method", "epsilon", "N",
    "artifact_seed", "gen_seed",
    "model_id",
    "decode_temperature", "decode_top_p", "max_new_tokens",
    "artifact_run_id", "artifact_dir",
    "output_text",
    "aux_score",
    "aux_ref",   # path to candidates JSON for ours, empty otherwise
]
KEY_COLS = [
    "prompt_hash","method","epsilon","N","artifact_seed","gen_seed",
    "decode_temperature","decode_top_p","max_new_tokens","artifact_run_id"
]
ensure_csv_schema(QUAL_CSV, COLS)
existing_keys = load_existing_keys(QUAL_CSV, KEY_COLS)

rows_to_add = []
def maybe_add_row(row: dict):
    key = tuple(str(row.get(c, "")) for c in KEY_COLS)
    if key in existing_keys:
        return False
    existing_keys.add(key)
    rows_to_add.append(row)
    return True

## Prepare output tables

In [ ]:
for eps in EPS_LIST:
    print("\n" + "#"*110)
    print(f"### EPS={eps} ###")
    print("#"*110)

    # Load RM for this eps (keep only within eps loop)
    rm_info = artifact["rm"][eps]
    rm = load_rm_head_only(rm_info["dir"], tokenizer, dtype=torch.float16)
    rm.eval()

    # Pre-load baseline models one by one per eps to avoid PEFT in-place issues
    # (A) DP-DPO policy model for eps
    dpo_info = artifact["dpo"][eps]
    dpo_model = None
    try:
        base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
        base.eval()
        adapter_dir = os.path.join(dpo_info["dir"], "adapter")
        dpo_model = PeftModel.from_pretrained(base, adapter_dir).to(DEVICE)
        dpo_model.eval()
    except Exception as e:
        cleanup(dpo_model)
        cleanup(base)
        cleanup(rm)
        raise RuntimeError(f"Failed loading dp_dpo adapter for eps={eps} at {dpo_info['dir']}") from e

    # (B) DP-PPO-like policy model for eps
    ppo_info = artifact["ppo"][eps]
    ppo_model = None
    try:
        base2 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
        base2.eval()
        adapter_dir2 = os.path.join(ppo_info["dir"], "adapter")
        ppo_model = PeftModel.from_pretrained(base2, adapter_dir2).to(DEVICE)
        ppo_model.eval()
    except Exception as e:
        cleanup(ppo_model)
        cleanup(base2)
        cleanup(dpo_model)
        cleanup(base)
        cleanup(rm)
        raise RuntimeError(f"Failed loading dp_ppo_like_split adapter for eps={eps} at {ppo_info['dir']}") from e

    for pid, (ph, prompt_raw) in enumerate(prompts):
        prompt = maybe_guard_prompt(prompt_raw)

        # print prompt preview (not full)
        preview = prompt_raw.replace("\n", " ")[:220]
        print("\n" + "="*110)
        print(f"[PROMPT {pid}] hash={ph}")
        print(f"preview: {preview}...")

        # ----- ours: best-of-N grid
        for N in N_GRID:
            gen_seed = GEN_SEED_BASE + 100000*pid + 1000*int(eps*10) + N

            comps, scores, best_idx = ours_best_of_n(pi0, rm, tokenizer, prompt, N=N, seed=gen_seed)
            best_text = comps[best_idx]
            best_score = float(scores[best_idx])

            # Save ALL candidates + scores (for sensitivity analysis)
            cand_path = os.path.join(
                CAND_DIR,
                f"ours_eps{eps}",
                f"N{N}",
                f"prompt{pid}_{ph}_seed{gen_seed}.json"
            )
            payload = {
                "prompt_id": int(pid),
                "prompt_hash": ph,
                "epsilon": float(eps),
                "N": int(N),
                "gen_seed": int(gen_seed),
                "decode": {"temperature": float(TEMPERATURE), "top_p": float(TOP_P), "max_new_tokens": int(MAX_NEW_TOKENS)},
                "rm_run_id": rm_info["run_id"],
                "rm_artifact_dir": rm_info["dir"],
                "best_idx": int(best_idx),
                "best_score": float(best_score),
                "candidates": [
                    {"cand_id": int(j), "rm_score": float(scores[j]), "text": comps[j]}
                    for j in range(len(comps))
                ],
            }
            save_candidates_json(cand_path, payload)

            # Print selected output
            print("\n" + "."*90)
            print(f"[ours | eps={eps} | N={N} | seed={gen_seed} | best_rm={best_score:.4f}]")
            print(best_text)

            row = dict(
                timestamp=now_ts(),
                prompt_id=int(pid),
                prompt_hash=ph,
                method="ours_bestofN",
                epsilon=float(eps),
                N=int(N),
                artifact_seed=int(ARTIFACT_SEED),
                gen_seed=int(gen_seed),
                model_id=MODEL_ID,
                decode_temperature=float(TEMPERATURE),
                decode_top_p=float(TOP_P),
                max_new_tokens=int(MAX_NEW_TOKENS),
                artifact_run_id=rm_info["run_id"],
                artifact_dir=rm_info["dir"],
                output_text=best_text,
                aux_score=float(best_score),
                aux_ref=cand_path,
            )
            maybe_add_row(row)

        # ----- dp_dpo: one generation
        gen_seed = GEN_SEED_BASE + 200000*pid + 1000*int(eps*10) + 7
        dpo_out = generate_completion(dpo_model, tokenizer, prompt, seed=gen_seed)
        print("\n" + "."*90)
        print(f"[dp_dpo | eps={eps} | seed={gen_seed}]")
        print(dpo_out)

        row = dict(
            timestamp=now_ts(),
            prompt_id=int(pid),
            prompt_hash=ph,
            method="dp_dpo",
            epsilon=float(eps),
            N=np.nan,
            artifact_seed=int(ARTIFACT_SEED),
            gen_seed=int(gen_seed),
            model_id=MODEL_ID,
            decode_temperature=float(TEMPERATURE),
            decode_top_p=float(TOP_P),
            max_new_tokens=int(MAX_NEW_TOKENS),
            artifact_run_id=dpo_info["run_id"],
            artifact_dir=dpo_info["dir"],
            output_text=dpo_out,
            aux_score=np.nan,
            aux_ref="",
        )
        maybe_add_row(row)

        # ----- dp_ppo_like_split: one generation
        gen_seed = GEN_SEED_BASE + 300000*pid + 1000*int(eps*10) + 9
        ppo_out = generate_completion(ppo_model, tokenizer, prompt, seed=gen_seed)
        print("\n" + "."*90)
        print(f"[dp_ppo_like_split | eps={eps} | seed={gen_seed}]")
        print(ppo_out)

        row = dict(
            timestamp=now_ts(),
            prompt_id=int(pid),
            prompt_hash=ph,
            method="dp_ppo_like_split",
            epsilon=float(eps),
            N=np.nan,
            artifact_seed=int(ARTIFACT_SEED),
            gen_seed=int(gen_seed),
            model_id=MODEL_ID,
            decode_temperature=float(TEMPERATURE),
            decode_top_p=float(TOP_P),
            max_new_tokens=int(MAX_NEW_TOKENS),
            artifact_run_id=ppo_info["run_id"],
            artifact_dir=ppo_info["dir"],
            output_text=ppo_out,
            aux_score=np.nan,
            aux_ref="",
        )
        maybe_add_row(row)

    # cleanup per eps to control GPU memory
    cleanup(ppo_model, base2)
    cleanup(dpo_model, base)
    cleanup(rm)

## Generate qualitative comparisons

In [ ]:
append_rows_csv(QUAL_CSV, rows_to_add, COLS)

print("\n" + "="*110)
print(f"[DONE] Appended {len(rows_to_add)} new rows to:")
print("  ", QUAL_CSV)
print("Candidate dumps saved under:")
print("  ", CAND_DIR)
print("="*110)

## Mixed-temperature Best-of-N

This workflow uses the same base policy with low- and high-temperature sampling, scores the pooled candidates with the private reward model, and records the selected response.

In [ ]:
import os, re, json, time, csv, hashlib, gc
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification

## Output paths

In [ ]:
ROOT = globals().get("ROOT", "/content/drive/MyDrive/Private_Finetuning")
CACHE_ROOT = os.path.join(ROOT, "cache")
os.makedirs(ROOT, exist_ok=True)
os.makedirs(CACHE_ROOT, exist_ok=True)

MASTER_CSV = os.path.join(ROOT, "master_results.csv")       # where dp_rm_postproc is recorded
QUAL_CSV   = os.path.join(ROOT, "qual_ours_mixedT.csv")     # ours-only results
CAND_DIR   = os.path.join(ROOT, "qual_candidates_mixedT")   # candidate dumps
os.makedirs(CAND_DIR, exist_ok=True)

print("ROOT      =", ROOT)
print("MASTER_CSV=", MASTER_CSV)
print("QUAL_CSV  =", QUAL_CSV)
print("CAND_DIR  =", CAND_DIR)

## Hugging Face cache

In [ ]:
HF_CACHE = os.path.join(CACHE_ROOT, "hf")
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = HF_CACHE

## Mixed-temperature configuration

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "GPU required."

MODEL_ID = "google/gemma-2b-it"

# Reward-model seed used for artifact selection
RM_SEED_FOR_ARTIFACT = 11

EPS_LIST = [0.5, 1.0, 2.0]
N_GRID   = [2, 4, 8, 16, 32]  # total candidates
T_LOW    = 0.2
T_HIGH   = 0.8
TOP_P    = 0.9
MAX_NEW_TOKENS = 160

# Prompt recovery (by hash) from hh-rlhf
N_TOTAL = 40000
TEST_FRAC = 0.2
PROMPT_SOURCE_SEED = 11

PROMPT_HASHES = [
    "6af5546e8a73",
    "e0966f2cec3d",
    "4461aba094ae",
    "bc1d9cb75240",
    "89a6a5b5c6a2",
]

# Printing behavior (NO REDACTION)
PRINT_PROMPTS = True
PRINT_ALL_BEST = True  # print best completion for each (eps, N)
PRINT_CANDIDATE_SAMPLES = False  # if True: prints a few candidates too

# RM scoring max length (match your RM training MAX_LEN=256)
RM_MAX_LEN = 256

# Resume key
RUN_TAG = f"mixedT_Tlow{T_LOW}_Thigh{T_HIGH}_topP{TOP_P}_maxNew{MAX_NEW_TOKENS}"

print("DEVICE =", DEVICE)
print("MODEL_ID =", MODEL_ID)
print("RM_SEED_FOR_ARTIFACT =", RM_SEED_FOR_ARTIFACT)
print("RUN_TAG =", RUN_TAG)

## Utility functions

In [ ]:
def cleanup(*objs):
    for o in objs:
        try: del o
        except: pass
    gc.collect()
    torch.cuda.empty_cache()

PROMPT_PAT = re.compile(r"\n\nAssistant:\s*")
def extract_prompt_prefix(text: str) -> str:
    ms = list(PROMPT_PAT.finditer(text))
    if not ms:
        return ""
    return text[: ms[-1].end()]

def sha1_12(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()[:12]

def now_ts() -> str:
    return time.strftime("%Y-%m-%d %H:%M:%S")

def ensure_csv_schema(path: str, cols: list):
    if os.path.exists(path):
        df = pd.read_csv(path)
        for c in cols:
            if c not in df.columns:
                df[c] = np.nan
        df.to_csv(path, index=False, quoting=csv.QUOTE_ALL)
    else:
        pd.DataFrame(columns=cols).to_csv(path, index=False, quoting=csv.QUOTE_ALL)

def load_existing_keys(path: str, key_cols: list):
    if not os.path.exists(path):
        return set()
    df = pd.read_csv(path)
    keys = set()
    for _, r in df.iterrows():
        keys.add(tuple(str(r.get(c, "")) for c in key_cols))
    return keys

def append_rows_csv(path: str, rows: list, cols: list):
    if not rows:
        return
    df_new = pd.DataFrame(rows, columns=cols)
    if os.path.exists(path):
        df_old = pd.read_csv(path)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new
    df.to_csv(path, index=False, quoting=csv.QUOTE_ALL)

## Load the tokenizer and base policy

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token_id = tokenizer.pad_token_id

pi0 = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, cache_dir=HF_CACHE).to(DEVICE)
pi0.eval()

## Locate reward-model artifacts

In [ ]:
dfm = pd.read_csv(MASTER_CSV)

def latest_ok_rm_artifact(df, eps: float, seed: int):
    sub = df[
        (df["method"].astype(str) == "dp_rm_postproc") &
        (pd.to_numeric(df["epsilon_target"], errors="coerce") == float(eps)) &
        (pd.to_numeric(df["seed"], errors="coerce") == int(seed)) &
        (df["train_status"].astype(str) == "ok") &
        (df["eval_status"].astype(str) == "ok")
    ].copy()
    if sub.empty:
        return None
    sub["ts"] = pd.to_datetime(sub["timestamp"], errors="coerce")
    sub = sub.sort_values(["ts", "run_id"], ascending=True)
    row = sub.iloc[-1]
    return str(row["run_id"]), str(row["artifact_dir"])

rm_art = {}
for eps in EPS_LIST:
    got = latest_ok_rm_artifact(dfm, eps=eps, seed=RM_SEED_FOR_ARTIFACT)
    if got is None:
        raise RuntimeError(f"No successful dp_rm_postproc found for eps={eps}, seed={RM_SEED_FOR_ARTIFACT} in {MASTER_CSV}")
    rm_run_id, rm_dir = got
    rm_art[eps] = {"run_id": rm_run_id, "dir": rm_dir}

print("\n[OK] RM artifacts (dp_rm_postproc):")
for eps in EPS_LIST:
    print(f"  eps={eps} | run_id={rm_art[eps]['run_id']} | dir={rm_art[eps]['dir']}")

## Load private reward models

In [ ]:
def load_rm_head_only(artifact_dir: str, dtype=torch.float16) -> AutoModelForSequenceClassification:
    meta_path = os.path.join(artifact_dir, "rm_head_meta.json")
    head_path = os.path.join(artifact_dir, "rm_head.pt")
    with open(meta_path, "r") as f:
        meta = json.load(f)
    head_attr = meta["head_attr"]
    model_id = meta["model_id"]

    rm = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=1, torch_dtype=dtype, cache_dir=HF_CACHE
    ).to(DEVICE)
    rm.config.pad_token_id = tokenizer.pad_token_id
    for p in rm.parameters():
        p.requires_grad_(False)

    head = getattr(rm, head_attr)
    sd = torch.load(head_path, map_location="cpu")
    head.load_state_dict(sd)
    rm.eval()
    return rm

## Recover fixed evaluation prompts

In [ ]:
raw = load_dataset("Anthropic/hh-rlhf", split=f"train[:{N_TOTAL}]", cache_dir=HF_CACHE)
split = raw.train_test_split(test_size=float(TEST_FRAC), shuffle=True, seed=int(PROMPT_SOURCE_SEED))
test_raw = split["test"]
train_raw = split["train"]

targets = set(PROMPT_HASHES)
found = {}

def scan_split(ds):
    for i in range(len(ds)):
        p = extract_prompt_prefix(ds[i]["chosen"])
        if not p:
            continue
        h = sha1_12(p)
        if h in targets and h not in found:
            found[h] = p
        if len(found) == len(targets):
            return

scan_split(test_raw)
if len(found) < len(targets):
    scan_split(train_raw)

missing = sorted(list(targets - set(found.keys())))
if missing:
    raise RuntimeError("Could not recover all prompts by hash. Missing: " + ", ".join(missing))

prompts = [(h, found[h]) for h in PROMPT_HASHES]
print("\n[OK] Loaded prompts by hash (no redaction):")
for j, (h, p) in enumerate(prompts):
    print(f"  prompt_id={j} hash={h} chars={len(p)}")

## Generate mixed-temperature candidates

In [ ]:
@torch.no_grad()
def generate_candidates(model, prompt: str, n: int, temperature: float, top_p: float, max_new: int, seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)

    inp = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    input_len = inp["input_ids"].shape[1]

    out = model.generate(
        **inp,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        num_return_sequences=n,
        max_new_tokens=max_new,
        pad_token_id=tokenizer.eos_token_id,
    )  # shape (n, total_len)

    comps = []
    for i in range(out.shape[0]):
        comp_ids = out[i, input_len:]
        comps.append(tokenizer.decode(comp_ids, skip_special_tokens=True).lstrip())
    return comps

@torch.no_grad()
def score_candidates(rm, prompt: str, comps: list):
    texts = [prompt + c for c in comps]
    enc = tokenizer(
        texts, return_tensors="pt",
        padding=True, truncation=True, max_length=RM_MAX_LEN
    ).to(DEVICE)
    scores = rm(**enc).logits.squeeze(-1).detach().float().cpu().numpy()
    return scores

## Prepare output tables

In [ ]:
COLS = [
    "timestamp",
    "prompt_id","prompt_hash",
    "epsilon","N_total","N_low","N_high",
    "T_low","T_high","top_p","max_new_tokens",
    "rm_run_id","rm_artifact_dir",
    "pi0_model_id",
    "gen_seed_base",
    "best_idx","best_score",
    "best_text",
    "cand_json_path",
    "run_tag",
]
KEY_COLS = ["prompt_hash","epsilon","N_total","rm_run_id","run_tag"]

ensure_csv_schema(QUAL_CSV, COLS)
existing = load_existing_keys(QUAL_CSV, KEY_COLS)

rows_to_add = []

def maybe_add_row(row):
    key = tuple(str(row.get(c, "")) for c in KEY_COLS)
    if key in existing:
        return False
    existing.add(key)
    rows_to_add.append(row)
    return True

## Run mixed-temperature Best-of-N

In [ ]:
for eps in EPS_LIST:
    rm = load_rm_head_only(rm_art[eps]["dir"], dtype=torch.float16)
    rm_run_id = rm_art[eps]["run_id"]
    rm_dir = rm_art[eps]["dir"]

    print("\n" + "#"*110)
    print(f"### OURS | eps={eps} | rm_run_id={rm_run_id} ###")
    print("#"*110)

    for pid, (ph, prompt) in enumerate(prompts):
        print("\n" + "="*110)
        print(f"[PROMPT {pid}] hash={ph}")
        if PRINT_PROMPTS:
            print(prompt)

        for N in N_GRID:
            N_low = N // 2
            N_high = N - N_low

            # deterministic base seed per (eps, pid, N)
            gen_seed_base = 2026 + 100000*pid + 1000*int(eps*10) + N

            # generate from two temperatures
            comps_low  = generate_candidates(pi0, prompt, n=N_low,  temperature=T_LOW,  top_p=TOP_P, max_new=MAX_NEW_TOKENS, seed=gen_seed_base + 1)
            comps_high = generate_candidates(pi0, prompt, n=N_high, temperature=T_HIGH, top_p=TOP_P, max_new=MAX_NEW_TOKENS, seed=gen_seed_base + 2)

            comps = comps_low + comps_high
            temps = [T_LOW]*len(comps_low) + [T_HIGH]*len(comps_high)

            scores = score_candidates(rm, prompt, comps)
            best_idx = int(np.argmax(scores))
            best_score = float(scores[best_idx])
            best_text = comps[best_idx]

            # save candidates json (all raw)
            cand_path = os.path.join(
                CAND_DIR,
                f"eps{eps}",
                f"N{N}",
                f"prompt{pid}_{ph}_seed{gen_seed_base}.json"
            )
            os.makedirs(os.path.dirname(cand_path), exist_ok=True)
            payload = {
                "prompt_id": int(pid),
                "prompt_hash": ph,
                "epsilon": float(eps),
                "N_total": int(N),
                "N_low": int(N_low),
                "N_high": int(N_high),
                "temps": {"T_low": float(T_LOW), "T_high": float(T_HIGH), "top_p": float(TOP_P)},
                "max_new_tokens": int(MAX_NEW_TOKENS),
                "pi0_model_id": MODEL_ID,
                "rm_run_id": rm_run_id,
                "rm_artifact_dir": rm_dir,
                "gen_seed_base": int(gen_seed_base),
                "best_idx": int(best_idx),
                "best_score": float(best_score),
                "prompt_text": prompt,  # raw
                "candidates": [
                    {
                        "cand_id": int(i),
                        "temp": float(temps[i]),
                        "rm_score": float(scores[i]),
                        "text": comps[i],
                    }
                    for i in range(len(comps))
                ],
            }
            with open(cand_path, "w") as f:
                json.dump(payload, f, indent=2)

            if PRINT_ALL_BEST:
                print("\n" + "."*90)
                print(f"[eps={eps} | N={N} | N_low={N_low}, N_high={N_high} | best_rm={best_score:.4f}]")
                print(best_text)

            if PRINT_CANDIDATE_SAMPLES:
                show = min(3, len(comps))
                print("  (sample candidates)")
                for i in range(show):
                    print(f"   - cand{i} temp={temps[i]} score={scores[i]:.4f} text[:80]={comps[i][:80]!r}")

            row = dict(
                timestamp=now_ts(),
                prompt_id=int(pid),
                prompt_hash=ph,
                epsilon=float(eps),
                N_total=int(N),
                N_low=int(N_low),
                N_high=int(N_high),
                T_low=float(T_LOW),
                T_high=float(T_HIGH),
                top_p=float(TOP_P),
                max_new_tokens=int(MAX_NEW_TOKENS),
                rm_run_id=str(rm_run_id),
                rm_artifact_dir=str(rm_dir),
                pi0_model_id=MODEL_ID,
                gen_seed_base=int(gen_seed_base),
                best_idx=int(best_idx),
                best_score=float(best_score),
                best_text=best_text,
                cand_json_path=cand_path,
                run_tag=RUN_TAG,
            )
            maybe_add_row(row)

    cleanup(rm)

append_rows_csv(QUAL_CSV, rows_to_add, COLS)
print("\n" + "="*110)
print(f"[DONE] appended {len(rows_to_add)} new rows -> {QUAL_CSV}")
print(f"candidate dumps -> {CAND_DIR}")
print("="*110)